In [1]:
# 🔹 1. Import Libraries
import numpy as np
import pandas as pd
import re
import nltk

nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

# 🔹 2. Load Dataset
dataset = pd.read_csv('Restaurant_Reviews.tsv', delimiter='\t', quoting=3)

# 🔹 3. Text Cleaning
corpus = []
ps = PorterStemmer()

for i in range(0, 1000):
    review = re.sub('[^a-zA-Z]', ' ', dataset['Review'][i])
    review = review.lower()
    review = review.split()
    
    all_stopwords = stopwords.words('english')
    if 'not' in all_stopwords:
        all_stopwords.remove('not')
    
    review = [ps.stem(word) for word in review if not word in set(all_stopwords)]
    review = ' '.join(review)
    corpus.append(review)

# 🔹 4. Bag of Words
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=1500)
X = cv.fit_transform(corpus).toarray()
y = dataset.iloc[:, -1].values

# 🔹 5. Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

# 🔹 6. Apply GridSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB

model = GaussianNB()

# Parameter grid
param_grid = {
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

# 🔹 Best Results
print("Best Parameters:", grid.best_params_)
print("Best Accuracy:", grid.best_score_)

# 🔹 7. Use Best Model
best_model = grid.best_estimator_

# 🔹 8. Prediction
y_pred = best_model.predict(X_test)

# 🔹 9. Evaluation
from sklearn.metrics import confusion_matrix, accuracy_score

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

accuracy = accuracy_score(y_test, y_pred)
print("Final Accuracy:", accuracy)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sowja\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Best Parameters: {'var_smoothing': 1e-05}
Best Accuracy: 0.6824999999999999
Confusion Matrix:
 [[56 41]
 [12 91]]
Final Accuracy: 0.735
